# E9 Phase A data/features — 2026-10-24 14:44
Intent: build candidate-preserving features using only historical positive overlaps and permissible schedules. Target viewing contributes only to the separate truth table.
Method: compile the module; test thresholds, session deduplication, deterministic channel ties, missing categories, unseen shows and candidate preservation on concrete toy data; load the real 16→17–20 split and build a history/schedule-selected sample. No model execution or environment changes.

In [1]:
from pathlib import Path
import subprocess, sys, time
import numpy as np
import pandas as pd
root = Path.cwd() if (Path.cwd() / 'artifacts').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'artifacts/scripts'))
from e9_data import load_split, build_features
result = subprocess.run([sys.executable, '-m', 'py_compile', str(root / 'artifacts/scripts/e9_data.py')], capture_output=True, text=True)
print('py_compile exit:', result.returncode); assert result.returncode == 0, result.stderr
h = pd.DataFrame({'user_id':['u']*5, 'tv_show_id':[1,1,2,3,1], 'week':[13,16,16,15,16], 'user_watch_pct':[.3,.5,.8,.2,.9], 'user_watch_seconds':[3,5,8,2,0], 'channel_id':[10,10,10,20,10], 'view_start':pd.to_datetime(['2026-03-23','2026-04-13','2026-04-13','2026-04-06','2026-04-13'])})
h['view_stop'] = h.view_start + pd.Timedelta(minutes=30); h.attrs['train_end'] = 16
future = pd.DataFrame({'tv_show_id':[1,1,1,1,2,4], 'channel_id':[20,10,20,10,20,20], 'duration':[10,10,10,10,20,50], 'start_time':pd.date_range('2026-04-20', periods=6, freq='h'), 'tv_show_category':['news']*6, 'tv_show_genre_1':[None]*6, 'tv_show_genre_2':['g2']*6, 'tv_show_genre_3':[None]*6})
past = future.iloc[[0]].copy()
c = pd.DataFrame({'user_id':['u']*4, 'tv_show_id':[4,1,99,2], 'lightfm_score':[.1,.4,.2,.3], 'lightfm_rank':[4,1,3,2], 'lightfm_user_bias':[.1]*4, 'lightfm_item_bias':[.2]*4, 'candidate_rank':[4,1,3,2]}, index=[8,3,9,1])
print('Toy before:', c[['user_id','tv_show_id']].to_dict('records'))
f = build_features(h, past, future, c); rows = f.set_index('tv_show_id')
pd.testing.assert_frame_equal(f[c.columns], c)
assert rows.loc[1, ['user_show_freq_03','user_show_freq_05','user_show_freq_08','recent_freq_1w','recent_freq_4w']].tolist() == [2,1,0,1,2]
assert rows.loc[2, ['user_show_freq_03','user_show_freq_05','user_show_freq_08']].tolist() == [1,1,1]
assert rows.loc[1, 'user_show_rank'] == rows.loc[2, 'user_show_rank'] == 1
assert rows.loc[1, 'user_channel_count'] == 2 and np.isclose(rows.loc[1,'user_channel_share'], 2/3)
assert rows.loc[2, 'user_channel_count'] == 1 and rows.loc[4,'future_past_airtime_ratio'] == 50
assert rows.loc[1, 'future_past_airtime_ratio'] == 4 and rows.loc[99,'category'] == '__MISSING__'
assert f.genre_1.eq('__MISSING__').all() and not f.duplicated(['user_id','tv_show_id']).any()
expected = set(c.columns) | {'user_show_freq_03','user_show_freq_05','user_show_freq_08','recent_freq_1w','recent_freq_4w','user_show_rank','user_channel_count','user_channel_share','category','genre_1','genre_2','genre_3','past_airtime','future_airtime','future_past_airtime_ratio'}
assert set(f.columns) == expected and np.isfinite(f.future_past_airtime_ratio).all()
print('Toy after:', f[['tv_show_id','user_show_freq_05','user_channel_count','future_past_airtime_ratio']].to_dict('records')); print('Toy checks: PASS; shape:', f.shape)
started = time.monotonic(); history, past_schedule, future_schedule, truth = load_split(16,17,20)
assert history.week.between(11,16).all() and past_schedule.start_time.dt.isocalendar().week.between(11,16).all()
assert future_schedule.start_time.dt.isocalendar().week.between(17,20).all()
assert all(s.duration.gt(0).all() and s.tv_show_id.ne(0).all() for s in (past_schedule,future_schedule))
assert not truth.duplicated(['user_id','tv_show_id']).any() and truth.user_id.map(type).eq(str).all()
assert pd.api.types.is_integer_dtype(truth.tv_show_id)
print('Real shapes history/past/future/truth:', history.shape, past_schedule.shape, future_schedule.shape, truth.shape)
print('Real history weeks:', int(history.week.min()), int(history.week.max()))
sample = pd.MultiIndex.from_product([history.user_id.unique()[:10], future_schedule.tv_show_id.unique()[:20]], names=['user_id','tv_show_id']).to_frame(index=False)
for column in c.columns.difference(['user_id','tv_show_id']): sample[column] = 0
real = build_features(history,past_schedule,future_schedule,sample)
pd.testing.assert_frame_equal(real[sample.columns], sample)
assert set(real.columns) == expected and np.isfinite(real.future_past_airtime_ratio).all()
assert not real.duplicated(['user_id','tv_show_id']).any()
print('Real feature sample:', real.shape, '; checks: PASS; seconds:', round(time.monotonic()-started,2))


py_compile exit: 0
Toy before: [{'user_id': 'u', 'tv_show_id': 4}, {'user_id': 'u', 'tv_show_id': 1}, {'user_id': 'u', 'tv_show_id': 99}, {'user_id': 'u', 'tv_show_id': 2}]
Toy after: [{'tv_show_id': 4, 'user_show_freq_05': 0, 'user_channel_count': 1, 'future_past_airtime_ratio': 50.0}, {'tv_show_id': 1, 'user_show_freq_05': 1, 'user_channel_count': 2, 'future_past_airtime_ratio': 4.0}, {'tv_show_id': 99, 'user_show_freq_05': 0, 'user_channel_count': 0, 'future_past_airtime_ratio': 0.0}, {'tv_show_id': 2, 'user_show_freq_05': 1, 'user_channel_count': 1, 'future_past_airtime_ratio': 20.0}]
Toy checks: PASS; shape: (4, 22)
Real shapes history/past/future/truth: (1455511, 14) (68380, 8) (46929, 8) (99694, 2)
Real history weeks: 11 16
Real feature sample: (200, 22) ; checks: PASS; seconds: 2.29


## Result interpretation
Compilation and all toy/real assertions passed (exit 0). The real split contains 1,455,511 history rows, 68,380 past schedule rows, 46,929 future schedule rows, and 99,694 unique truth pairs. Loading plus the 200-row feature sample took 2.29 seconds. Candidate rows, order, index and supplied scores were preserved, with exactly 15 added features. Candidate rows were selected independently of truth. The toy verified threshold boundaries, session deduplication across shows, the lower-channel-ID deterministic tie, zero-overlap exclusion, missing categories, and finite ratios for future-only shows. No failed attempts, reverted changes, dependency installs, or model executions.